# Task 2.1 (part 2): Beyond-accuracy metrics

Metrics that measure something other than hitting the held-out items: how diverse, novel and calibrated
each user's list is, how exposure is spread over the catalogue, and whether item groups and user
groups are treated alike. Like the accuracy metrics in [task_2_1.ipynb](task_2_1.ipynb), they are
implemented in NumPy in [metrics.py](metrics.py) (section *Beyond-accuracy metrics*), so every model
is scored by the same code.

**Conventions**

* Every metric takes the top-K list $R_K(u)$ of each user. Per-user metrics are averaged over users.
* $\text{pop}(i)$ = number of distinct **training** users who interacted with item $i$ (Task 1 split).
* Genres are the 19 binary genres of `u.item` (`class` field of RecBole's `ml-100k.item`, including `unknown`).
  Every item has at least one genre.
* Items are integer indices `0..n_items-1`. `recs` is an int array `[n_users, K]`, best first, and
  `histories` holds each user's training items in the same row order.

**Layout**

1. The metrics, with their definitions
2. Sanity checks on hand-worked examples
3. The metrics on the Task 2.2 top-10 lists (Random, Popularity and the six tuned models)

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.spatial.distance import jensenshannon

from metrics import (HEAD, MID, TAIL, K, aplt, arp, catalog_coverage, evaluate_beyond_accuracy,
                     evaluate_topk, exposure_counts, exposure_gini, gini, group_exposure, gru,
                     ild, item_popularity, js_divergence, miscalibration, novelty,
                     popularity_bins, short_head_mask, upd)
from recbole_utils import ROOT, load_split

## 1. The metrics

| # | Metric | Function | Definition | Better |
|---|---|---|---|---|
| 1 | ILD@K | `ild` | mean over pairs $i<j$ in $R_K(u)$ of $1-\cos(\mathbf g_i,\mathbf g_j)$ (genre vectors) | higher |
| 2 | Novelty@K | `novelty` | mean over $R_K(u)$ of $-\log_2\frac{\text{pop}(i)+1}{n_\text{users}+1}$ | higher |
| 3 | MC_KL@K | `miscalibration` | $\sum_g p(g\mid u)\log\frac{p(g\mid u)}{\tilde q(g\mid u)}$, skipping $p(g\mid u)=0$ | lower |
| 4 | Coverage@K | `catalog_coverage` | $\lvert\bigcup_u R_K(u)\rvert / n_\text{items}$ | higher |
| 5 | Gini@K | `exposure_gini` | Gini index of how often each catalogue item is recommended (0 if never) | lower |
| 6 | Group exposure@K | `group_exposure` | $\frac{1}{\lvert G_k\rvert}\sum_u\sum_{i\in R_K(u)\cap G_k}\frac{1}{\log_2(1+\text{pos}(i))}$ for short head / long tail, and their ratio | ratio → 1 |
| 7 | GRU | `gru` | $\lvert \overline{F}_{G_1} - \overline{F}_{G_2}\rvert$ for per-user accuracy $F$ | lower |
| 8 | UPD@K | `upd` | $\text{JSD}_2$ between head/mid/tail shares of the training profile and of $R_K(u)$ | lower |
| 9 | ARP@K, APLT@K | `arp`, `aplt` | mean $\text{pop}(i)/n_\text{users}$; share of long-tail items in $R_K(u)$ | lower ARP / higher APLT = less popularity bias |

`evaluate_beyond_accuracy(...)` computes all of them for one model and returns a summary and the
per-user values.

**Details and choices**

* **Miscalibration** (Steck, 2018). $p(g\mid i)$ is item $i$'s genre vector normalised to sum to 1.
  $p(g\mid u)$ averages it uniformly over the user's training history. $q(g\mid u)$ averages it over
  $R_K(u)$ with rank weights $w_r(i) = 1/\log_2(\text{rank}(i)+1)$. $\tilde q = (1-\alpha)q + \alpha p$
  with $\alpha = 0.01$ keeps the KL finite when the list misses a genre the user watches. Natural log.
* **Gini** of a sorted count vector $x_{(1)}\le\dots\le x_{(n)}$:
  $\frac{\sum_j (2j-n-1)\,x_{(j)}}{n\sum_j x_j}$. The maximum, everything on one item, is $(n-1)/n$.
* **Short head / long tail** (metrics 6 and 9). The short head is the top 20% of the catalogue by
  $\text{pop}$ (336 of 1,682 items; ties broken by item index), the long tail is the rest. APLT uses
  the same split.
* **Group exposure** is a sum over users, divided by the group size. Both values are the expected rank-discounted
  exposure *per item* of the group. The ratio short head / long tail is 1 at parity.
* **GRU** user groups: gender (from `u.user`; group 1 = female) and activity (group 1 = the 20% of users
  with the largest training profiles). $F$ = NDCG@10.
* **UPD head/mid/tail** are cut by *interaction share*, not item count. The head is the most
  popular items until together they hold 20% of training interactions, the tail is the least popular
  items until they hold 20%, and the mid is the rest. The item at each boundary goes to the head or tail. JSD is in bits, so it lies in [0, 1].

## 2. Sanity checks

### 2.1 Hand-worked examples

Toy catalogue of 6 items over 3 genres:

| item | genres | pop |
|---|---|---|
| 0 | A | 9 |
| 1 | A | 5 |
| 2 | B | 3 |
| 3 | A, B | 1 |
| 4 | C | 0 |
| 5 | A, B, C | 2 |

with $n_\text{users} = 15$.

In [2]:
G = np.array([[1, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0], [0, 0, 1], [1, 1, 1]])
pop = np.array([9, 5, 3, 1, 0, 2])
n_users = 15

# --- ILD: same genre -> 0, disjoint genres -> 1, A vs A+B -> 1 - 1/sqrt(2)
assert np.allclose(ild([[0, 1]], G), 0)
assert np.allclose(ild([[0, 2]], G), 1)
assert np.allclose(ild([[0, 3]], G), 1 - 1 / np.sqrt(2))
# three items: pairs (0,1)=0, (0,2)=1, (1,2)=1 -> 2/3
assert np.allclose(ild([[0, 1, 2]], G), 2 / 3)

# --- Novelty: pop 0 with 15 users -> -log2(1/16) = 4; pop 15 -> 0
assert np.allclose(novelty([[4]], pop, n_users), 4)
assert np.allclose(novelty([[4, 0]], pop, n_users), (4 + -np.log2(10 / 16)) / 2)

# --- Miscalibration
# every list item has the user's genre mix -> 0 (history all A, list all A)
assert np.allclose(miscalibration([[0, 1]], [[0, 1]], G), 0, atol=1e-12)
# history all A, list all B: p = (1,0,0), q = (0,1,0), q~(A) = 0.01 -> KL = log(100)
assert np.allclose(miscalibration([[2]], [[0, 1]], G), np.log(100))
# rank weights: history (A, B) -> p = (.5, .5, 0); list [A, B] weights 1 and 1/log2(3)
w = np.array([1, 1 / np.log2(3)]); q = np.array([w[0], w[1], 0]) / w.sum()
qt = 0.99 * q + 0.01 * np.array([.5, .5, 0])
expected = np.sum(.5 * np.log(.5 / qt[:2]))
assert np.allclose(miscalibration([[0, 2]], [[0, 2]], G), expected)
assert np.allclose(miscalibration([[0, 2]], [[1, 2]], G), expected)  # p only depends on the genres of the history

# --- Coverage and exposure Gini, catalogue of 6
recs = np.array([[0, 1], [0, 2], [0, 1]])
assert catalog_coverage(recs, 6) == 3 / 6
assert exposure_counts(recs, 6).tolist() == [3, 2, 1, 0, 0, 0]
assert np.isclose(gini(np.ones(6)), 0)
assert np.isclose(gini([0, 0, 0, 0, 0, 7]), 5 / 6)

# --- Short head / long tail: top 20% of 6 items = round(1.2) = 1 item (item 0)
head = short_head_mask(pop)
assert head.tolist() == [True, False, False, False, False, False]

# --- Group exposure: lists [[0, 1], [0, 2], [0, 1]], head = {0} at rank 1 three times
# head: 3 * 1 / 1 item = 3; tail: 3 * (1/log2 3) over 5 items
exp = group_exposure(recs, head)
assert np.isclose(exp["short_head"], 3) and np.isclose(exp["long_tail"], 3 / np.log2(3) / 5)

# --- ARP / APLT
assert np.allclose(arp([[0, 4]], pop, n_users), (9 / 15 + 0) / 2)
assert np.allclose(aplt([[0, 4], [1, 2]], ~head), [0.5, 1])

# --- GRU
assert np.isclose(gru([0.2, 0.4, 0.1, 0.1], [True, True, False, False]), 0.2)
print("hand-worked examples OK")

hand-worked examples OK


### 2.2 Head / mid / tail bins and UPD

`pop = [9, 5, 3, 1, 0, 2]` has 20 interactions. In popularity order the items are 0 (45%), 1 (25%),
2 (15%), 5 (10%), 3 (5%), 4 (0%).

* Head: item 0 alone already holds 45% ≥ 20%, so head = {0}.
* Tail: from the bottom, 4 (0%), 3 (5%), 5 (10%) together hold 15% < 20%. Adding item 2 reaches 30%,
  so 2 is the boundary item and goes to the tail too: tail = {4, 3, 5, 2}.
* Mid = {1}.

UPD: a list with the same head/mid/tail shares as the profile gives 0. A list with disjoint bins gives 1 (the
maximum, in bits). The JSD is also checked against SciPy (`jensenshannon` returns its square root).

In [3]:
bins = popularity_bins(pop)
assert bins.tolist() == [HEAD, MID, TAIL, TAIL, TAIL, TAIL]

assert np.allclose(upd([[0, 2]], [[0, 4, 3, 5]], bins), js_divergence([.25, 0, .75], [.5, 0, .5]))
assert np.allclose(upd([[0, 1, 2]], [[0, 1, 4]], bins), 0)   # same shares, different items
assert np.allclose(upd([[0]], [[2, 3]], bins), 1)            # all head vs. all tail

rng = np.random.default_rng(0)
for _ in range(200):
    p, q = rng.dirichlet(np.ones(3)), rng.dirichlet(np.ones(3))
    p[rng.integers(3)] = 0; p /= p.sum()  # include zero entries
    assert np.isclose(js_divergence(p, q), jensenshannon(p, q, base=2) ** 2)

# Gini against the mean-absolute-difference definition
for _ in range(200):
    x = rng.integers(0, 20, rng.integers(2, 50))
    if x.sum() == 0:
        continue
    mad = np.abs(x[:, None] - x[None, :]).sum() / (2 * len(x) ** 2 * x.mean())
    assert np.isclose(gini(x), mad)

# ILD against a pairwise loop on random lists
genres = (rng.random((40, 5)) < 0.4).astype(int); genres[genres.sum(1) == 0, 0] = 1
r = rng.choice(40, size=(30, 6), replace=True)
cos = lambda a, b: a @ b / np.linalg.norm(a) / np.linalg.norm(b)
loop = [np.mean([1 - cos(genres[l[i]], genres[l[j]]) for i in range(6) for j in range(i + 1, 6)]) for l in r]
assert np.allclose(ild(r, genres), loop)
print("bins / UPD / JSD / Gini / ILD checks OK")

bins / UPD / JSD / Gini / ILD checks OK


## 3. On the Task 2.2 top-10 lists

The top-10 test lists from [task_2_2.ipynb](task_2_2.ipynb) (`results/task_2_2/recommendations/`,
external ids) are scored here. The Task 1 split is rebuilt to get the training interactions, which
give popularity and the user histories. Items are indexed by RecBole's internal id minus 1
(id 0 is padding), so the catalogue is all 1,682 items, including ones nobody rated in training.

For GRU, NDCG@10 is recomputed from the lists and the test items with `evaluate_topk`. It must match
the per-user NDCG@10 saved by Task 2.2, which checks the id mapping.

In [4]:
TASK22 = ROOT / "group_18" / "results" / "task_2_2"
RESULTS_DIR = ROOT / "group_18" / "results" / "task_2_1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS = ["Random", "Popularity", "BPR", "FISM", "ItemKNN", "UserKNN", "SLIMElastic", "EASE"]

_, dataset, train, _, test = load_split("Pop")
UID, IID = dataset.uid_field, dataset.iid_field
N_ITEMS = dataset.item_num - 1
tr_u, tr_i = train.dataset.inter_feat[UID].numpy(), train.dataset.inter_feat[IID].numpy() - 1
N_USERS = len(np.unique(tr_u))

pop = item_popularity(tr_u, tr_i, N_ITEMS)
history = pd.Series(tr_i).groupby(tr_u).apply(np.asarray)
te = test.dataset.inter_feat
test_items = pd.Series(te[IID].numpy() - 1).groupby(te[UID].numpy()).apply(set).to_dict()

# genre matrix from the atomic item file, rows in internal-id order
item_file = pd.read_csv(ROOT / "dataset" / "ml-100k" / "ml-100k.item", sep="\t", dtype=str)
genre_lists = item_file.set_index("item_id:token")["class:token_seq"].str.split()
genre_lists.index = dataset.token2id(IID, genre_lists.index.to_numpy()) - 1
genre_lists = genre_lists.sort_index()
assert genre_lists.index.tolist() == list(range(N_ITEMS))
GENRES = sorted({g for gs in genre_lists for g in gs})
genres = np.array([[g in gs for g in GENRES] for gs in genre_lists], dtype=int)

# user groups, per internal user id
users_file = pd.read_csv(ROOT / "dataset" / "ml-100k" / "ml-100k.user", sep="\t", dtype=str)
gender = pd.Series(users_file["gender:token"].to_numpy(),
                   index=dataset.token2id(UID, users_file["user_id:token"].to_numpy()))
profile_size = history.map(len)
active = profile_size >= profile_size.quantile(0.8)  # top 20% by training profile size

print(f"{N_USERS} users, {N_ITEMS} items, {len(tr_u)} training interactions, {len(GENRES)} genres")
print(f"female users: {(gender == 'F').sum()}, active users: {active.sum()} "
      f"(training profile >= {profile_size[active].min()} items)")
print(f"short head: {short_head_mask(pop).sum()} items; "
      f"UPD bins head/mid/tail: {np.bincount(popularity_bins(pop), minlength=3).tolist()} items")

943 users, 1682 items, 80808 training interactions, 19 genres
female users: 273, active users: 189 (training profile >= 138 items)
short head: 336 items; UPD bins head/mid/tail: [58, 473, 1151] items


In [5]:
def load_recs(name):
    df = pd.read_csv(TASK22 / "recommendations" / f"{name}_top{K}.csv", dtype=str)
    df["user"] = dataset.token2id(UID, df["user_id"].to_numpy())
    df["item"] = dataset.token2id(IID, df["item_id"].to_numpy()) - 1
    df["rank"] = df["rank"].astype(int)
    wide = df.pivot(index="user", columns="rank", values="item").sort_index()
    return wide.index.to_numpy(), wide.to_numpy()


rows, per_user = {}, {}
for name in MODELS:
    users, recs = load_recs(name)
    acc = evaluate_topk(dict(zip(users, recs)), {u: test_items[u] for u in users}, ks=(K,))["per_user"]
    ndcg = acc.loc[users, f"ndcg@{K}"].to_numpy()
    saved = pd.read_csv(TASK22 / "per_user" / f"{name}.csv", index_col=0).loc[users, f"ndcg@{K}"]
    assert np.allclose(ndcg, saved), f"{name}: NDCG@{K} differs from Task 2.2"

    summary, pu = evaluate_beyond_accuracy(
        recs, [history[u] for u in users], genres, pop, N_USERS,
        user_groups={"gender": (gender.loc[users] == "F").to_numpy(), "activity": active.loc[users].to_numpy()},
        accuracy=ndcg)
    rows[name] = pd.concat([pd.Series({f"ndcg@{K}": ndcg.mean()}), summary])
    per_user[name] = pu.set_index(users)
    per_user[name].to_csv(RESULTS_DIR / f"beyond_accuracy_per_user_{name}.csv")

beyond = pd.DataFrame(rows).T.sort_values(f"ndcg@{K}", ascending=False)
beyond.to_csv(RESULTS_DIR / "beyond_accuracy.csv")
print("NDCG@10 matches Task 2.2 for every model")

NDCG@10 matches Task 2.2 for every model


### 3.1 Results

Test top 10 for each model, sorted by NDCG@10. Section 1 lists the better direction of each metric.

In [6]:
beyond.round(3)

,ndcg@10,ild@10,novelty@10,mc_kl@10,upd@10,arp@10,aplt@10,coverage@10,gini@10,exposure_head@10,exposure_tail@10,exposure_ratio@10,gru_gender,gru_activity
EASE,0.330,0.728,2.105,0.707,0.161,0.251,0.014,0.237,0.918,12.613,0.035,365.556,0.044,0.242
SLIMElastic,0.311,0.734,2.047,0.688,0.179,0.259,0.007,0.198,0.928,12.685,0.017,763.490,0.034,0.233
UserKNN,0.288,0.726,1.983,0.805,0.205,0.270,0.005,0.174,0.940,12.706,0.011,1106.551,0.047,0.193
ItemKNN,0.282,0.731,1.974,0.776,0.227,0.268,0.005,0.154,0.945,12.703,0.012,1034.329,0.040,0.199
FISM,0.243,0.719,2.313,0.791,0.136,0.220,0.032,0.240,0.918,12.421,0.083,150.388,0.042,0.225
BPR,0.235,0.714,2.534,0.783,0.097,0.206,0.134,0.432,0.856,11.189,0.390,28.680,0.014,0.164
Popularity,0.142,0.751,1.435,0.852,0.552,0.373,0.000,0.037,0.987,12.752,0.000,inf,0.039,0.131
Random,0.011,0.772,5.640,1.198,0.324,0.047,0.823,0.995,0.240,2.287,2.612,0.875,0.001,0.021


### 3.2 Observations

* **Accuracy comes with heavy popularity bias.** The four most accurate models (EASE, SLIMElastic,
  UserKNN, ItemKNN) take 99% or more of every list from the short head (APLT@10 ≤ 0.014), cover only
  15–24% of the catalogue, and have an exposure Gini above 0.9. A short-head item gets 365–1,100 times the
  rank-discounted exposure of a long-tail item. Popularity is the extreme case: 62 items in total, no tail.
* **BPR is the least biased model.** It has 13% long-tail items, 43% coverage, Gini 0.86 and an
  exposure ratio of 29. It also has the lowest UPD (0.097), so its lists match each user's own
  head/mid/tail mix best. This costs about 0.09 NDCG@10 against EASE.
* **Popularity taste (UPD) separates the models more than genre diversity (ILD).** ILD@10 stays between
  0.71 and 0.77 for every list, Random included. Over 19 coarse genres, most pairs of films already differ.
  Personalised lists are a little *less* genre-diverse than Popularity, because they focus on the
  genres the user watches.
* **Calibration follows accuracy.** EASE and SLIMElastic have the lowest MC_KL@10 (0.69–0.71).
  Popularity (0.85) and Random (1.20) ignore the user's genre mix.
* **User-side unfairness.** Gender: women get lower NDCG@10 under every model (e.g. EASE 0.299 vs.
  0.342), with a gap of 0.01–0.05. Activity: the 20% most active users get much higher NDCG@10 (EASE 0.523 vs.
  0.281). Part of this gap is mechanical: they have 4.5 times as many test items (26.8 vs. 6.0), and that
  raises NDCG@10 even for a non-personalised ranking (Popularity: 0.246 vs. 0.115).